# R2 — ResNet-18 Double Descent

**Experiment**: Validate DD in a deeper residual architecture (WideResNet18).  
**Dataset**: Same CIFAR-10 subset as R1 (n=5000, η=15%, seed=42).  
**Sweep**: k ∈ {1, 2, 4, 8, 16, 32}, 1 seed → 6 runs.  
**Optimizer**: SGD + momentum + cosine-LR decay (standard ResNet recipe).  
**Output**: `results/R2/fig2_r2_dd.png`

> **Estimated time**: ~40–60 min/run on Colab T4 → ~4 h total.


## 1  Environment setup

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

REPO_URL  = 'https://github.com/YOUR_USERNAME/project-6699.git'  # ← replace
REPO_DIR  = '/content/project-6699'
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/R2'
else:
    RESULT_DIR = f'{REPO_DIR}/results/R2'

os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)

## 2  (Optional) Run a subset of widths

R2 has only 6 points but each run is ~40–60 min.  Split across two accounts if needed.

In [ ]:
from run_r2 import R2_CONFIG, run_r2, plot_r2
import copy

cfg = copy.deepcopy(R2_CONFIG)

# ── Full sweep (default) ─────────────────────────────────────────────────────
# cfg['widths'] = [1, 2, 4, 8, 16, 32]

# ── Account A: narrow ────────────────────────────────────────────────────────
# cfg['widths'] = [1, 2, 4]

# ── Account B: wide ──────────────────────────────────────────────────────────
# cfg['widths'] = [8, 16, 32]

print(f"Will train {len(cfg['widths']) * len(cfg['seeds'])} runs: k={cfg['widths']}")

## 3  Run R2

In [ ]:
import threading, time
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js
            eval_js('0')
        except Exception:
            pass
_t = threading.Thread(target=_keep_alive, daemon=True)
_t.start()

results = run_r2(cfg, RESULT_DIR, resume=True)

## 4  Plot Figure 2

In [ ]:
plot_r2(RESULT_DIR)

from pathlib import Path
from IPython.display import Image
fig_path = Path(RESULT_DIR) / 'fig2_r2_dd.png'
if fig_path.exists():
    display(Image(str(fig_path)))

## 5  Summary table

In [ ]:
import pandas as pd
from src.io_utils import load_results

results = load_results(RESULT_DIR, pattern='r2_*.json')
df = pd.DataFrame([{
    'k':           r['width_multiplier'],
    'n_params':    r['n_params'],
    'train_err':   f"{r['train_error']:.3f}",
    'test_err':    f"{r['test_error']:.3f}",
    'wall_time_m': f"{r['wall_time_s']/60:.1f}",
} for r in results])
df = df.sort_values('k')
display(df.to_string(index=False))